In [15]:
# Import the built-in Operating System (os) module
import os
import pandas as pd
import numpy as np
import sqlite3

# Import the drive utility from the google.colab package
from google.colab import drive

# Mount the drive to the specific path '/content/drive'
# 'force_remount=True' ensures it reconnects if it was already mounted in a previous run
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [17]:
# Change the Current Working Directory to your project folder
project_dir = "/content/drive/MyDrive/Projects/personalNotes/"
os.chdir(project_dir)

# Verify the change
print(f"New CWD: {os.getcwd()}")

New CWD: /content/drive/MyDrive/Projects/personalNotes


# 📘 [Introduction](https://learn.microsoft.com/en-us/training/modules/introduction-to-transact-sql/1-introduction/?ns-enrollment-type=learningpath&ns-enrollment-id=learn.wwl.get-started-querying-with-transact-sql)

## 1. What is SQL?
- **Definition**: SQL stands for **Structured Query Language**.
- **Purpose**: It is the standard language used to communicate with, manage, and query **relational databases**.
- **Core Tasks**:
  - Retrieve data (e.g., using the `SELECT` statement).
  - Update data (e.g., using `INSERT`, `UPDATE`, `DELETE`).
- **Common RDBMS**: Microsoft SQL Server, MySQL, PostgreSQL, MariaDB, Oracle.
- **Standardization**: Governed by the **ANSI** (American National Standards Institute) standard, though vendors add their own proprietary extensions.

---

## 2. Transact-SQL (T-SQL)
- **What is it?**: T-SQL is Microsoft's proprietary dialect of SQL used in SQL Server, Azure SQL Database, and Microsoft Fabric.
- **Extensions**: It adds functionality beyond the ANSI standard, including:
  - **Programmability**: Writing stored procedures and functions (application code stored directly inside the database).
  - **Security**: Managing user accounts, roles, and permissions.
  - **Control Flow**: Variables, loops, and error handling.

---

## 3. SQL: A Declarative Language
Programming languages generally fall into two categories:

| Type | Description | Example |
| :--- | :--- | :--- |
| **Procedural** | You define the exact *sequence of steps* the computer must follow to achieve a result. | C, Java, Python |
| **Declarative** | You describe *what output you want*, and the engine figures out *how* to get it. | **SQL**, HTML |

### How SQL works under the hood:
1. You write a query describing the desired result.
2. The database engine's **Query Processor** evaluates the request.
3. It uses data **statistics** and **indexes** to generate an optimal **Query Plan**.
4. The engine executes the plan to retrieve the data efficiently.

---

## 4. Relational Data Model
SQL is primarily used to interact with **relational databases**, where data is organized into **tables** (technically referred to as *relations*).

### Key Concepts:
- **Table (Relation)**: Represents a specific entity type (e.g., `Customer`, `Product`, `SalesOrder`).
- **Column (Attribute)**: Represents a specific characteristic of the entity (e.g., `Name`, `Price`, `OrderDate`).
- **Row (Record/Instance)**: Represents a single, specific occurrence of the entity (e.g., *John Doe*, *iPhone 14*, *Order #12345*).

### Keys and Relationships:
- **Primary Key (PK)**: A column (or combination of columns) that *uniquely identifies* each row in a table.
- **Foreign Key (FK)**: A column in one table that *references* the Primary Key of another table, establishing a link between them.

### Example Schema:
```mermaid
erDiagram
    CUSTOMER ||--o{ SALESORDERHEADER : places
    SALESORDERHEADER ||--|{ SALESORDERDETAIL : contains
    PRODUCT ||--o{ SALESORDERDETAIL : "is ordered in"

    CUSTOMER {
        int CustomerID PK
        string Name
    }
    SALESORDERHEADER {
        int OrderID PK
        int CustomerID FK
        date OrderDate
    }
    SALESORDERDETAIL {
        int OrderID PK,FK
        int LineItemNo PK
        int ProductID FK
        int Quantity
    }
    PRODUCT {
        int ProductID PK
        string ProductName
        decimal Price
    }
```

## 5. Set-Based Processing
Relational databases are built on the mathematical foundation of **Set Theory**.

- **What is a Set?**: A collection of definite, distinct objects considered as a whole. In SQL, a table (like `Customer`) represents a set of all customers. The result of a `SELECT` query is also a set.
- **The Set-Based Mindset**:
  - ❌ **Row-by-Row (Procedural)**: "Get the first row, process it, get the next row..."
  - ✅ **Set-Based (Declarative)**: "Apply this operation to *all* rows that meet this condition *all at once*."
- **Crucial Rule: No Inherent Order**:
  - In set theory, a set has **no specific order**.
  - Therefore, a relational table has no concept of a "first row" or "last row". Elements can be accessed or retrieved in any order.
  - If you need data returned in a specific sequence, you **must** explicitly define it using the `ORDER BY` clause in your `SELECT` query.

---


# 🗂️ [Working with Schemas in SQL Server](https://learn.microsoft.com/en-us/training/modules/introduction-to-transact-sql/2-work-with-schemas/?ns-enrollment-type=learningpath&ns-enrollment-id=learn.wwl.get-started-querying-with-transact-sql)

In SQL Server (and many other relational database systems), **schemas** are used to organize and group database objects into logical namespaces.

---

## 1. What is a Schema?
A **schema** is a logical container within a database that holds database objects such as tables, views, indexes, and stored procedures.

### Why use Schemas?
1. **Logical Grouping:** Organize objects by business function, department, or application module.
2. **Disambiguation:** Allow multiple tables with the **exact same name** to exist in the same database without naming conflicts.
3. **Security & Permissions:** Make it easier to grant or revoke access to a whole group of objects at the schema level.

---

## 2. Real-World Example: Disambiguation
Imagine a retail business database. The company needs to track two completely different types of "Orders":
1. **Customer Orders:** Placed by customers buying finished goods.
2. **Supplier Orders:** Placed by the procurement team buying raw components.

Instead of naming them `CustomerOrder` and `SupplierOrder`, we can use schemas to group them logically:

| Schema (Namespace) | Tables Contained | Purpose |
| :--- | :--- | :--- |
| **`Sales`** | `Customer`, `Order` | Tracks customer-facing transactions. |
| **`Production`** | `Product`, `Order` | Tracks manufacturing and supplier component orders. |

Because they are in different schemas, the database engine treats `Sales.Order` and `Production.Order` as two completely distinct tables.

### Visualizing the Structure
```mermaid
graph TD
    DB[(Database: StoreDB)]
    
    DB --> S1(Schema: Sales)
    DB --> S2(Schema: Production)
    
    S1 --> T1[Table: Customer]
    S1 --> T2[Table: Order]
    
    S2 --> T3[Table: Product]
    S2 --> T4[Table: Order]
    
    style DB fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style S1 fill:#fff3e0,stroke:#ef6c00
    style S2 fill:#e8f5e9,stroke:#2e7d32
```

---

## 3. Hierarchical Naming System
SQL Server uses a strict hierarchical naming convention to locate objects.

### The Fully Qualified Name (4-Part Name)
To uniquely identify any object across an entire enterprise environment, SQL Server uses a 4-part naming structure:

```sql
[Server_Name].[Database_Name].[Schema_Name].[Object_Name]
```

**Example:**
```sql
Server1.StoreDB.Sales.Order
```
* **`Server1`**: The database server instance.
* **`StoreDB`**: The specific database.
* **`Sales`**: The schema (namespace).
* **`Order`**: The table (object).

### The Common Name (2-Part Name)
When you are already connected to a specific database (which is the case 99% of the time when writing queries), you don't need to specify the server or database name. It is standard practice to use the **2-part name**:

```sql
[Schema_Name].[Object_Name]
```

**Example:**
```sql
SELECT * FROM Sales.Order;
SELECT * FROM Production.Order;
```

---

## 💡 Pro-Tip: The `dbo` Default Schema
If you create a table and **do not** specify a schema, SQL Server automatically places it in the default schema, which is usually **`dbo`** (Database Owner).

```sql
-- This creates a table named 'NewTable' inside the 'dbo' schema
CREATE TABLE NewTable (ID INT);

-- The fully qualified name is actually:
-- dbo.NewTable
```
*Best Practice:* Always explicitly specify the schema (e.g., `Sales.NewTable`) in your scripts to avoid ambiguity and ensure objects are created in the correct namespace.

---

## 📝 Quick Summary

| Concept | Description | Example |
| :--- | :--- | :--- |
| **Schema** | A logical namespace grouping database objects. | `Sales`, `Production` |
| **Disambiguation** | Using schemas to allow identical table names. | `Sales.Order` vs `Production.Order` |
| **4-Part Name** | Fully qualified name across servers/databases. | `Server1.StoreDB.Sales.Order` |
| **2-Part Name** | Standard naming used within a single database. | `Sales.Order` |
| **Default Schema** | Where objects go if no schema is specified. | `dbo` |

---

## Explore the Structure of SQL Statements

In any SQL dialect, statements are categorized into distinct groups based on their core function.

### Core SQL Statement Types

* **DML (Data Manipulation Language):** Focuses on querying and modifying the actual data within tables. This is the most common category used for daily operations.
  * **Key Statements:** `SELECT`, `INSERT`, `UPDATE`, `DELETE`
  * **Primary Use Cases:**
    * **Data Analysts:** Retrieve data using `SELECT` for reports and analysis.
    * **Application Developers:** Perform **CRUD** operations (Create, Read, Update, Delete) on application data.
* **DDL (Data Definition Language):** Handles the structure, definition, and life cycle of database objects themselves (such as tables, views, and stored procedures).
  * **Key Statements:** `CREATE`, `ALTER`, `DROP`
* **DCL (Data Control Language):** Used by administrators to manage security and user access permissions for various database objects.
  * **Key Statements:** `GRANT`, `REVOKE`, `DENY`

### Additional Categorizations
Depending on the documentation or platform, you might also see these terms used:

* **TCL (Transaction Control Language):** Used to manage and control transactions (e.g., ensuring a batch of operations either all succeed or all fail together).
* **DQL (Data Query Language):** Sometimes, DML is strictly redefined as "Data *Modification* Language" (which excludes querying). When this happens, `SELECT` statements get their own category called DQL.

# 🏗️ Structure of SQL Statements

In any SQL dialect, SQL statements are grouped into different categories based on their purpose. Understanding these categories helps in organizing database operations, managing security, and defining database structures.

---

## 🗂️ Overview of SQL Statement Types

```mermaid
mindmap
  root((SQL Statements))
    DML
      Data Manipulation Language
      Query & Modify Data
    DDL
      Data Definition Language
      Define Database Objects
    DCL
      Data Control Language
      Manage Security & Permissions
    TCL
      Transaction Control Language
      Manage Transactions
    DQL
      Data Query Language
      Query Data (SELECT)
```

---

## 1. 📊 Data Manipulation Language (DML)
**Purpose:** Focuses on querying and modifying the actual *data* stored within the database tables.
**Key Statements:**
- `SELECT`: Retrieves data (The primary focus of data analysis).
- `INSERT`: Adds new rows of data.
- `UPDATE`: Modifies existing data.
- `DELETE`: Removes data.

**Who uses it?**
- **Data Analysts:** To retrieve data for reports and analysis.
- **App Developers:** To perform **CRUD** operations in applications.

---

## 2. 🏛️ Data Definition Language (DDL)
**Purpose:** Handles the definition, structure, and life cycle of database *objects* (like tables, views, indexes, and procedures). It changes the schema of the database rather than the data inside it.
**Key Statements:**
- `CREATE`: Builds a new database object (e.g., `CREATE TABLE`).
- `ALTER`: Modifies the structure of an existing object (e.g., adding a new column).
- `DROP`: Deletes an entire database object from the database.

---

## 3. 🔐 Data Control Language (DCL)
**Purpose:** Manages security, access rights, and permissions for users and database objects.
**Key Statements:**
- `GRANT`: Gives a user permission to perform specific tasks.
- `REVOKE`: Removes previously granted permissions.
- `DENY`: Explicitly prevents a user from performing a specific task (overrides GRANT).

---

## 4. 🔄 Additional Classifications (TCL & DQL)
Depending on the specific SQL dialect or training material, you might encounter these additional categories:

- **TCL (Transaction Control Language):** Used to manage, control transactions and ensures data integrity (e.g., ensuring a batch of operations either all succeed or all fail together).
  - *Examples:* `COMMIT`, `ROLLBACK`, `SAVEPOINT`.
- **DQL (Data Query Language):** Sometimes, DML is strictly redefined as "Data *Modification* Language" (which excludes querying). When this happens, `SELECT` statements get their own category called DQL.

---

## 🔄 DML and CRUD Operations

**CRUD** is a concept used in software development, while DML is the SQL implementation of those operations.

| CRUD Operation | Meaning | SQL DML Statement |
| :--- | :--- | :--- |
| **C** | **Create** | `INSERT` |
| **R** | **Read** | `SELECT` |
| **U** | **Update** | `UPDATE` |
| **D** | **Delete** | `DELETE` |

```mermaid
flowchart LR

A[Software Application]

A --> B[Create]

A --> C[Read]

A --> D[Update]

A --> E[Delete]

B --> F["INSERT"]

C --> G["SELECT"]

D --> H["UPDATE"]

E --> I["DELETE"]
```

---

## 📝 Summary Cheat Sheet

| Category | Full Name | Purpose | Common Commands |
| :--- | :--- | :--- | :--- |
| **DML** | Data Manipulation Language | Query & modify data | `SELECT`, `INSERT`, `UPDATE`, `DELETE` |
| **DDL** | Data Definition Language | Define database structure | `CREATE`, `ALTER`, `DROP` |
| **DCL** | Data Control Language | Manage security/permissions | `GRANT`, `REVOKE`, `DENY` |
| **TCL** | Transaction Control Language | Manage transactions | `COMMIT`, `ROLLBACK` |
| **DQL** | Data Query Language | Query data (if separated from DML) | `SELECT` |

---

# 🔍 Examining the SELECT Statement

The `SELECT` statement is the most frequently used Data Manipulation Language (DML) statement in SQL. While Transact-SQL (T-SQL) is Microsoft's dialect of the ANSI standard SQL, the core mechanics of the `SELECT` statement apply broadly across relational databases.

> **⚠️ Crucial Concept:** The order in which you **write** a SQL query is *not* the order in which the database engine **executes** it. Understanding the **Logical Processing Order** is essential for writing correct, efficient queries and avoiding common errors.

---

## 1. 🔄 The Logical Processing Order

When you write a query, you start with `SELECT`. However, the database engine cannot select columns until it knows which table to get them from. Therefore, it processes clauses in a completely different sequence.

### 📝 Writing Order vs. 🏃 Execution Order

```mermaid
graph LR
    subgraph WRITE["📝 How You WRITE It (Syntax)"]
        direction LR
        W1[SELECT] --> W2[FROM] --> W3[WHERE] --> W4[GROUP BY] --> W5[HAVING] --> W6[ORDER BY]
    end

    subgraph RUN["🏃 How SQL RUNS It (Logic)"]
        direction LR
        E1[1. FROM] --> E2[2. WHERE] --> E3[3. GROUP BY] --> E4[4. HAVING] --> E5[5. SELECT] --> E6[6. ORDER BY]
    end
    
    style W1 fill:#ffcdd2,stroke:#c62828
    style E1 fill:#c8e6c9,stroke:#2e7d32
```

### Step-by-Step Execution Breakdown

Let's trace how the engine processes a complex query:

```sql
SELECT OrderDate, COUNT(OrderID) AS Orders
FROM Sales.SalesOrder
WHERE Status = 'Shipped'
GROUP BY OrderDate
HAVING COUNT(OrderID) > 1
ORDER BY OrderDate DESC;
```

| Step | Clause | Action Taken | Virtual Table State |
| :--- | :--- | :--- | :--- |
| **1** | **`FROM`** | Identifies the source table (`Sales.SalesOrder`). | Loads all rows and columns from the table into memory. |
| **2** | **`WHERE`** | Filters **individual rows** based on a predicate (`Status = 'Shipped'`). | Discards unshipped orders. Only columns from `FROM` are accessible here. |
| **3** | **`GROUP BY`** | Organizes the filtered rows into unique groups based on `OrderDate`. | Creates a new virtual table of "groups". *From here on, you can only use grouped columns or aggregate functions.* |
| **4** | **`HAVING`** | Filters **entire groups** based on an aggregate condition (`COUNT(OrderID) > 1`). | Discards groups (dates) that only have 1 or fewer orders. |
| **5** | **`SELECT`** | Determines the final columns to return. Calculates expressions and applies **aliases** (`AS Orders`). | Creates the final output shape. |
| **6** | **`ORDER BY`** | Sorts the final result set (`DESC`). | Returns the final formatted rowset to the user. |

### 🚨 The "Alias Trap" (Common Interview Question)
Because `SELECT` is evaluated **5th**, any column aliases created in the `SELECT` clause **do not exist yet** when the `WHERE`, `GROUP BY`, or `HAVING` clauses are processed.

❌ **This will FAIL:**
```sql
SELECT ListPrice - StandardCost AS Markup
FROM Production.Product
WHERE Markup > 100; -- ERROR: Invalid column name 'Markup'
```
✅ **This will WORK:**
```sql
SELECT ListPrice - StandardCost AS Markup
FROM Production.Product
WHERE (ListPrice - StandardCost) > 100; -- Repeat the expression
```
*(Note: `ORDER BY` is the only clause that **can** use aliases, because it executes after `SELECT`)*.

---

## 2. 🌟 Selecting Columns: `*` vs. Explicit Lists

### The Danger of `SELECT *` (The "Star")
Using `SELECT *` retrieves all columns from the source table. While fine for quick ad-hoc testing, it is **strictly discouraged in production code**.

| Issue | Description |
| :--- | :--- |
| **🔀 Schema Changes** | If a DBA adds, removes, or rearranges columns in the underlying table, your application or report might break or display data in the wrong columns. |
| **🐌 Performance Hits** | It forces the database to read, process, and transmit data over the network that your application doesn't even need. This wastes I/O, memory, and network bandwidth. |
| **🚫 Index Inefficiency** | `SELECT *` usually prevents the use of "Covering Indexes", forcing the engine to do expensive lookups to the actual data pages. |

```sql
-- ❌ Bad Practice for Production
SELECT * FROM Production.Product;

-- ✅ Good Practice
SELECT ProductID, Name, ListPrice, StandardCost
FROM Production.Product;
```

---

## 3. 🧮 Selecting Expressions

The `SELECT` clause isn't just for retrieving stored data; it can also perform **scalar calculations** (calculations that result in a single value per row).

### Types of Expressions
1. **Mathematical:** Using `+`, `-`, `*`, `/` on numeric columns.
2. **String Manipulation:** Using `+` to concatenate strings (in T-SQL).

```sql
SELECT ProductID,
       Name + ' (' + ProductNumber + ')' AS ProductDescription,
       ListPrice - StandardCost AS ProfitMargin
FROM Production.Product;
```

> **💡 Pro-Tip on Data Types:** Be very careful with the `+` operator. If you try to add a string and a number, SQL Server will attempt implicit conversion, which often results in an error (e.g., *Conversion failed when converting the varchar value to data type int*). Always ensure data types match the operation you intend to perform.

---

## 4. 🏷️ Specifying Column Aliases

When you select an expression, the resulting column often has no name (or an ugly default name like `(No column name)`). You use the **`AS`** keyword to assign a meaningful **alias**.

```sql
SELECT
    ProductID AS ID,
    Name + ' (' + ProductNumber + ')' AS ProductName,
    ListPrice - StandardCost AS Markup
FROM Production.Product;
```

*   **Is `AS` optional?** Yes, in T-SQL you can write `ProductID ID`, but using `AS` is a **best practice** for readability.
*   **Spacing in Aliases:** If you want an alias to contain spaces, you must wrap it in brackets `[]` or double quotes `""`.
    ```sql
    SELECT ListPrice - StandardCost AS [Profit Margin] ...
    ```

---

## 5. 🎨 Formatting Best Practices

SQL is generally case-insensitive and ignores whitespace, meaning the engine doesn't care how your code looks. However, **human readability** is critical for debugging and maintenance.

### 🏆 The Golden Rules of SQL Formatting:
1. **Capitalize Keywords**: Capitalize all T-SQL Keywords, e.g., SELECT, FROM, WHERE, AS, etc.
2. **Line Breaks**: Start a new line for each major clause.
3. **Column Lists**: If the SELECT list has many columns or complex expressions, put each column on its own line.
4. **Indentation**: Indent subclauses or column lists to visually group them under their parent clause.

```sql
SELECT
    p.ProductID,
    p.Name AS ProductName,
    p.ListPrice - p.StandardCost AS Markup,
    c.Name AS CategoryName
FROM Production.Product AS p
    INNER JOIN Production.ProductCategory AS c
        ON p.ProductCatID = c.CategoryID
WHERE
    p.ListPrice > 0
    AND c.Name LIKE '%Bikes%'
GROUP BY
    p.ProductID,
    p.Name,
    p.ListPrice,
    p.StandardCost,
    c.Name
ORDER BY
    Markup DESC;
```

### Checklist for Clean Code:
- [ ] Are keywords (`SELECT`, `FROM`, `WHERE`) capitalized?
- [ ] Is there a line break before every major clause?
- [ ] Are table aliases used to shorten long table names? (e.g., `FROM Production.Product AS p`)
- [ ] Are complex expressions indented so they stand out from standard columns?

---

## 📝 Summary & Quick Revision

```mermaid
mindmap
  root((SELECT<br/>Statement))
    Execution Order
      1. FROM (Source)
      2. WHERE (Row Filter)
      3. GROUP BY (Aggregation)
      4. HAVING (Group Filter)
      5. SELECT (Columns/Aliases)
      6. ORDER BY (Sorting)
    Column Selection
      Avoid SELECT *
      Explicit Columns
      Expressions (Math/String)
    Aliases
      Use AS keyword
      Rename expressions
      Cannot use in WHERE/HAVING
    Formatting
      Capitalize Keywords
      Line breaks per clause
      Indent for readability
```

### 🎯 Interview Cheat Sheet
*   **Q: Why can't I use a column alias in the `WHERE` clause?**
    *   **A:** Because of the logical processing order. `WHERE` is evaluated (Step 2) before `SELECT` (Step 5), so the alias doesn't exist yet when the filter is applied.
*   **Q: What is the difference between `WHERE` and `HAVING`?**
    *   **A:** `WHERE` filters **individual rows** *before* grouping. `HAVING` filters **aggregated groups** *after* the `GROUP BY` clause has been applied.
*   **Q: Why is `SELECT *` bad for production?**
    *   **A:** It causes network overhead, prevents covering index usage, and makes applications vulnerable to breaking if the underlying table schema changes.


# 🗃️ [Working with Data Types & Conversions in T-SQL](https://learn.microsoft.com/en-us/training/modules/introduction-to-transact-sql/5a-data-types)

In SQL Server (T-SQL), every column, variable, and expression has a **data type**. The data type dictates how the database engine stores the data and how it behaves in expressions (e.g., whether the `+` operator performs mathematical addition or string concatenation).

---

## 1. 📊 Common SQL Server Data Types

Understanding data types is crucial for writing efficient queries and avoiding conversion errors.

| Category | Common Data Types | Description / Use Case |
| :--- | :--- | :--- |
| **Exact Numeric** | `tinyint`, `smallint`, `int`, `bigint`, `decimal`, `numeric` | Whole numbers or exact decimals (e.g., money, counts). |
| **Approximate Numeric**| `float`, `real` | Scientific calculations where slight rounding is acceptable. |
| **Character (String)** | `char`, `varchar`, `nchar`, `nvarchar`, `text`, `ntext` | Text data. `n` prefixes denote Unicode (supports multiple languages). |
| **Date/Time** | `date`, `time`, `datetime`, `datetime2`, `datetimeoffset` | Storing dates and times. `datetime2` is the modern standard. |
| **Binary** | `binary`, `varbinary`, `image` | Storing raw bytes, files, or images. |
| **Other / Specialized** | `bit`, `uniqueidentifier`, `xml`, `json`, `geography` | Booleans, GUIDs, structured documents, spatial data. |

> 💡 **Pro-Tip:** Always choose the **smallest data type** that can safely hold your data. For example, use `tinyint` (1 byte) instead of `int` (4 bytes) for an `Age` column. This saves memory and improves index performance.

---

## 2. 🔄 Data Type Conversion

When you mix different data types in an expression, SQL Server must convert them to a common type.

### Implicit vs. Explicit Conversion
- **Implicit Conversion:** SQL Server automatically converts data types based on data type precedence.
  *(Example: Adding an `int` to a `decimal` results in a `decimal`.)*
- **Explicit Conversion:** You must tell SQL Server exactly how to convert the data using built-in functions. Failing to do so results in an error.
  *(Example: Concatenating a `varchar` and a `decimal` using `+` will throw an error unless you explicitly convert the `decimal` to a `varchar` first.)*

```mermaid
flowchart TD
    A[Mixing Data Types in Expression] --> B{Are they compatible?}
    B -->|Yes| C[Implicit Conversion<br/>SQL Server handles it automatically]
    B -->|No| D[Explicit Conversion<br/>You must use a function]
    
    C --> C1[Example: int + decimal<br/>Example: char + varchar]
    D --> D1[Example: varchar + decimal<br/>Example: string + date]
    
    style C fill:#c8e6c9,stroke:#2e7d32
    style D fill:#ffcdd2,stroke:#c62828
```


---

## 3. 🛠️ Core Conversion Functions

T-SQL provides several functions to handle explicit conversions. Choosing the right one depends on your needs for **standards compliance**, **formatting**, or **error handling**.

### A. `CAST` and `TRY_CAST`
`CAST` is the **ANSI SQL standard** function (supported across almost all RDBMS like PostgreSQL, MySQL, Oracle).

- **`CAST(expression AS data_type)`**: Converts the value. **Throws an error** if the conversion fails.
- **`TRY_CAST(expression AS data_type)`**: T-SQL specific. **Returns `NULL`** if the conversion fails, preventing the query from crashing.

```sql
-- 1. Standard CAST (Concatenating int and string)
SELECT CAST(ProductID AS varchar(4)) + ': ' + Name AS ProductName
FROM Production.Product;
-- Result: '680: HL Road Frame - Black, 58'

-- 2. TRY_CAST (Handling dirty data safely)
-- Suppose 'Size' contains '58', 'M', 'L'
SELECT TRY_CAST(Size AS int) AS NumericSize
FROM Production.Product;
-- Result: 58, NULL, NULL (Instead of throwing an error on 'M')
```

### B. `CONVERT` and `TRY_CONVERT`
`CONVERT` is **T-SQL specific** (not ANSI standard). Its superpower is the optional **`style`** parameter, which is incredibly useful for formatting dates and numbers into strings.

- **`CONVERT(data_type, expression, [style])`**

```sql
-- Formatting Dates using Style Codes
SELECT
    SellStartDate,
    CONVERT(varchar(20), SellStartDate) AS DefaultFormat,
    CONVERT(varchar(10), SellStartDate, 101) AS US_Format -- mm/dd/yyyy
FROM SalesLT.Product;

/* Result:
SellStartDate                | DefaultFormat      | US_Format
2002-06-01T00:00:00.0000000  | Jun 1 2002 12:00AM | 6/1/2002
*/
```
*(Note: Style `101` is US date format `mm/dd/yyyy`. Style `103` is British/French `dd/mm/yyyy`.)*

### C. `PARSE` and `TRY_PARSE`
`PARSE` is specifically designed to convert **formatted strings** (that include cultural formatting like currency symbols or specific date layouts) into numeric or date/time types. It relies on the .NET Framework CLR.

```sql
-- Parsing strings with symbols/formats
SELECT
    PARSE('01/01/2021' AS date) AS DateValue,
    PARSE('$199.99' AS money) AS MoneyValue;

/* Result:
DateValue                  | MoneyValue
2021-01-01 00:00:00.0000000| 199.99
*/
```
> ⚠️ **Performance Note:** `PARSE` is slower than `CAST` and `CONVERT`. Only use it when dealing with complex, culture-specific string formats.

### D. `STR`
The `STR` function is a quick way to convert a **numeric value to a character string** (varchar).

```sql
SELECT ProductID, '$' + STR(ListPrice, 10, 2) AS Price
FROM Production.Product;
-- Result: '680 | $   1432.00'
```

---

## 4. 🗺️ Which Function Should I Use?

```mermaid
mindmap
  root((Conversion<br/>Functions))
    CAST
      ANSI Standard
      Basic conversion
      Fails on error
    TRY_CAST
      T-SQL Specific
      Returns NULL on error
      Great for dirty data
    CONVERT
      T-SQL Specific
      Supports Style formatting
      Best for Dates/Numbers to String
    PARSE
      T-SQL Specific
      Parses culture-specific strings
      Slower performance
      E.g. '$100' or 'Jan 1, 2020'
    STR
      Number to String
      Quick formatting
```

---

## 5. 💻 Code Playground: Putting it Together

Here is a comprehensive example combining data types, conversions, and string manipulation.

```sql
-- Scenario: Creating a custom formatted report string
SELECT
    -- Original Data
    ProductID,
    ListPrice,
    SellStartDate,
    
    -- Explicit Conversions
    -- 1. Convert ID to string for concatenation
    CAST(ProductID AS varchar(10)) AS ProductID_String,
    
    -- 2. Format Date to 'YYYY-MM-DD' using CONVERT style 120
    CONVERT(varchar(10), SellStartDate, 120) AS FormattedDate,
    
    -- 3. Combine into a single readable sentence
    'Product #' + CAST(ProductID AS varchar(10)) +
    ' was released on ' +
    CONVERT(varchar(10), SellStartDate, 101) +
    ' at a price of $' +
    CAST(ListPrice AS varchar(20)) AS ReportString

FROM SalesLT.Product
WHERE ListPrice > 1000;
```

---

## 6. ⚠️ Common Pitfalls & Interview Tips

| Pitfall | Explanation | Solution |
| :--- | :--- | :--- |
| **The `+` Operator Ambiguity** | `+` means **Addition** for numbers, but **Concatenation** for strings. | Always explicitly convert numbers to strings before using `+` to concatenate. (e.g., `CAST(Price AS varchar)`). |
| **Implicit Conversion Performance** | If you filter a string column with a numeric value (e.g., `WHERE PhoneNumber = 12345`), SQL Server must implicitly convert the *entire column* row-by-row, destroying index usage (SARGability). | **Always match data types.** Use `WHERE PhoneNumber = '12345'`. |
| **TRY_CAST returning NULL** | Using `TRY_CAST` prevents crashes, but the resulting `NULL` might break downstream logic or aggregations. | Wrap `TRY_CAST` in `ISNULL()` or `COALESCE()` to provide a fallback value. <br>`ISNULL(TRY_CAST(Size AS int), 0)` |
| **Data Truncation** | Converting a long string to a `varchar(5)` will silently truncate the data without throwing an error. | Always ensure the target `varchar` length is large enough, or use `varchar(MAX)`. |

---

## 📝 Summary Checklist
- [ ] Know the difference between `int`, `decimal`, `varchar`, and `datetime`.
- [ ] Understand that `+` behaves differently based on data types.
- [ ] Use `CAST` for standard, safe conversions.
- [ ] Use `TRY_CAST` when dealing with messy, untrusted data.
- [ ] Use `CONVERT(..., style)` when you need specific Date/Number string formats.
- [ ] Avoid implicit conversions in `WHERE` clauses to maintain query performance (SARGability).

---


# 🕳️ Handling NULL Values in T-SQL

In relational databases, dealing with missing or unknown data is a critical skill. SQL provides specific functions and logical operators to identify, replace, and manipulate `NULL` values effectively.

---

## 1. 🌌 What is a NULL?

A `NULL` value represents **missing, unknown, or unappplied data**.
It is crucial to understand what `NULL` is **NOT**:
- ❌ It is **NOT** zero (`0`).
- ❌ It is **NOT** a blank or empty string (`''`).
- ❌ It is **NOT** a space (`' '`).

Zero and empty strings are known values. `NULL` means "we don't know what is here yet" (e.g., a customer hasn't provided their middle name or email address yet).

### The Concept of NULL
```mermaid
flowchart LR
    A[Data Value] --> B{Is it known?}
    B -->|Yes| C[Actual Value<br/>e.g., 0, '', 'John']
    B -->|No| D[NULL<br/>Unknown / Missing]
    
    style C fill:#c8e6c9,stroke:#2e7d32
    style D fill:#ffcdd2,stroke:#c62828
```

> ⚠️ **Crucial Rule:** `NULL` is not equal to anything, and it is not unequal to anything. You cannot use standard comparison operators (`=`, `<`, `>`) with `NULL`. To check for a `NULL`, you must use `IS NULL` or `IS NOT NULL`.

---

## 2. 🔄 The `ISNULL()` Function

The `ISNULL()` function is a **T-SQL specific** function used to replace a `NULL` with a specified default value.

### Syntax
```sql
ISNULL ( check_expression , replacement_value )
```
- **`check_expression`**: The expression to be checked for `NULL`.
- **`replacement_value`**: The value to return if `check_expression` is `NULL`.

### Example: Replacing Missing Middle Names
Suppose the `MiddleName` column contains `NULL`s for customers who don't have one. We can replace them with the string `'None'`.

```sql
SELECT FirstName,
       ISNULL(MiddleName, 'None') AS MiddleIfAny,
       LastName
FROM Sales.Customer;
```

**Result:**
| FirstName | MiddleIfAny | LastName |
| :--- | :--- | :--- |
| Orlando | N. | Gee |
| Keith | **None** | Howard |
| Donna | F. | Gonzales |

### ⚠️ Important Rules for `ISNULL`
1. **Data Type Matching:** The `replacement_value` must be of the same data type (or implicitly convertible to) the `check_expression`. You cannot replace a `varchar` `NULL` with an `int` like `0` without casting.
2. **Choose Placeholders Wisely:** Pick a replacement value (like `'None'` or `-1`) that you are absolutely sure will never appear as a legitimate, actual data value in your column.

---

## 3. 🏆 The `COALESCE()` Function

While `ISNULL` is limited to two arguments, **`COALESCE`** is the **ANSI SQL standard** function that can take a **variable number of arguments**. It evaluates them in order and returns the **first expression that is NOT `NULL`**.

### Syntax
```sql
COALESCE ( expression1, expression2, [ ,...n ] )
```

### How COALESCE Works
```mermaid
flowchart TD
    A[Start COALESCE] --> B{Is Exp1 NULL?}
    B -->|No| C[Return Exp1]
    B -->|Yes| D{Is Exp2 NULL?}
    D -->|No| E[Return Exp2]
    D -->|Yes| F{Is ExpN NULL?}
    F -->|No| G[Return ExpN]
    F -->|Yes| H[Return NULL]
    
    style C fill:#c8e6c9,stroke:#2e7d32
    style E fill:#c8e6c9,stroke:#2e7d32
    style G fill:#c8e6c9,stroke:#2e7d32
    style H fill:#ffcdd2,stroke:#c62828
```

### Example: Calculating Weekly Earnings
Imagine an `HR.Wages` table where employees are paid in one of three ways: Hourly, Weekly Salary, or Commission. Two of the three columns will be `NULL` for any given employee.

```sql
SELECT EmployeeID,
       COALESCE(HourlyRate * 40,
                WeeklySalary,
                Commission * SalesQty) AS WeeklyEarnings
FROM HR.Wages;
```

**Result:**
| EmployeeID | WeeklyEarnings |
| :--- | :--- |
| 1 | 899.76 *(Calculated from HourlyRate)* |
| 2 | 1001.00 *(Pulled from WeeklySalary)* |
| 3 | 1298.77 *(Calculated from Commission)* |

### `COALESCE` vs `ISNULL`
| Feature | `ISNULL` | `COALESCE` |
| :--- | :--- | :--- |
| **Standard** | T-SQL Specific (SQL Server) | **ANSI SQL Standard** (Works in Postgres, Oracle, MySQL) |
| **Arguments** | Exactly 2 | Multiple (2 to *n*) |
| **Data Type** | Returns the data type of `check_expression` | Evaluates data type precedence of all arguments |
| **Nullability** | Result is always considered `NOT NULL` (even if both inputs are nullable) | Result is considered `NULL` if all inputs are nullable |

---

## 4. 🎯 The `NULLIF()` Function

The `NULLIF()` function does the exact opposite of `ISNULL`. It allows you to **return `NULL` if two expressions are equal**. If they are not equal, it returns the **first expression**.

### Syntax
```sql
NULLIF ( expression1 , expression2 )
```

### Logic Flow
```mermaid
flowchart TD
    A[NULLIF Exp1, Exp2] --> B{Does Exp1 = Exp2?}
    B -->|Yes| C[Return NULL]
    B -->|No| D[Return Exp1]
    
    style C fill:#ffcdd2,stroke:#c62828
    style D fill:#c8e6c9,stroke:#2e7d32
```

### Example: Data Cleansing (Removing Placeholder Zeros)
Suppose a `Discount` column uses `0` as a placeholder for "No Discount applied", but your reporting system requires `NULL` to represent "No Discount" so it can be ignored in averages.

```sql
SELECT SalesOrderID,
       ProductID,
       UnitPrice,
       NULLIF(UnitPriceDiscount, 0) AS Discount
FROM Sales.SalesOrderDetail;
```

**Result:**
| SalesOrderID | ProductID | UnitPrice | Discount |
| :--- | :--- | :--- | :--- |
| 71774 | 836 | 356.898 | **NULL** *(Because 0 = 0)* |
| 71780 | 988 | 112.998 | 0.40 *(Because 0.40 != 0)* |
| 71781 | 748 | 818.700 | **NULL** *(Because 0 = 0)* |

### 💡 Pro-Tip: Preventing Divide-By-Zero Errors
`NULLIF` is famously used to prevent the dreaded *"Divide by zero error encountered"* in SQL.
Instead of writing:
```sql
SELECT TotalSales / NULLIF(SalesCount, 0) -- Wait, this still errors if not careful!
```
You write:
```sql
-- If SalesCount is 0, NULLIF returns NULL. Anything divided by NULL is NULL (no error!).
SELECT TotalSales / NULLIF(SalesCount, 0) AS AverageSale
```
*(Note: In T-SQL, dividing by NULL yields NULL, preventing the query from crashing).*

---

## 5. 📝 Summary Cheat Sheet

| Function | Purpose | Syntax | Standard |
| :--- | :--- | :--- | :--- |
| **`ISNULL`** | Replaces `NULL` with a specified value. | `ISNULL(expr, replacement)` | T-SQL Only |
| **`COALESCE`** | Returns the **first non-NULL** from a list. | `COALESCE(expr1, expr2, ...)` | **ANSI SQL** |
| **`NULLIF`** | Returns `NULL` if two expressions are **equal**. | `NULLIF(expr1, expr2)` | **ANSI SQL** |
| **`IS NULL`** | Predicate to check if a value is `NULL`. | `WHERE column IS NULL` | **ANSI SQL** |

---

## 6. 🎯 Interview Questions & Common Pitfalls

### ❌ Pitfall 1: Using `= NULL`
**Question:** What is wrong with `WHERE MiddleName = NULL`?
**Answer:** It will return **zero rows**. Because `NULL` represents an unknown value, it cannot be equal to anything (even another `NULL`). You must always use `WHERE MiddleName IS NULL`.

### ❌ Pitfall 2: Data Type Precedence in COALESCE
**Question:** What happens if you run `COALESCE(1, 'Text')`?
**Answer:** SQL Server will attempt to convert the string `'Text'` to an integer because `int` has a higher data type precedence than `varchar`. This will result in a **conversion error**. Always ensure all expressions in a `COALESCE` list are of compatible data types, or explicitly `CAST` them.

### 🧠 Interview Question: `ISNULL` vs `COALESCE`
**Interviewer:** "What is the difference between `ISNULL` and `COALESCE`?"
**You:**
1. `ISNULL` is T-SQL specific and takes exactly 2 arguments. `COALESCE` is ANSI standard, takes multiple arguments, and is portable across different RDBMS.
2. `COALESCE` evaluates data type precedence among all arguments, whereas `ISNULL` uses the data type of the first argument.
3. `ISNULL` treats the result as `NOT NULL` (if the replacement is not null), while `COALESCE` evaluates the nullability of all expressions and can return a nullable result.

---


# Exercise: Getting Started with Transact-SQL

## 1. Basic SELECT Queries
The `SELECT` statement is the primary tool for querying tables.

* **Retrieve All Columns:** Use `*` (asterisk) to pull every column.

    ```sql
    SELECT * FROM SalesLT.Product;
    ```
* **Retrieve Specific Columns:** List the exact columns you want, separated by commas.
    ```sql
    SELECT Name, StandardCost, ListPrice FROM SalesLT.Product;
    ```
* **Calculated Columns:** You can perform math directly in the `SELECT` list. By default, this new column will not have a name.
    ```sql
    SELECT Name, ListPrice - StandardCost FROM SalesLT.Product;
    ```
* **Column Aliases (`AS`):** Assign a readable name to calculated columns (or rename existing ones) using the `AS` keyword.
    ```sql
    SELECT Name AS ProductName, ListPrice - StandardCost AS Markup
    FROM SalesLT.Product;
    ```
* **String Concatenation:** Use the `+` operator to join text strings together.
    ```sql
    SELECT ProductNumber, Color, Size, Color + ', ' + Size AS ProductDetails
    FROM SalesLT.Product;
    ```

---

## 2. Working with Data Types
The behavior of operators like `+` depends entirely on the data type. If you mix text and numbers, you must explicitly convert them to avoid errors.

* **`CAST`:** The ANSI standard way to convert data types.
    ```sql
    -- Converts the numeric ProductID to a varchar so it can be concatenated with text
    SELECT CAST(ProductID AS varchar(5)) + ': ' + Name AS ProductName
    FROM SalesLT.Product;
    ```
* **`CONVERT`:** SQL Server's specific conversion function. It behaves like `CAST` but includes an optional third parameter for formatting (especially useful for dates).
    ```sql
    -- The '126' formats the date to the ISO8601 standard
    SELECT SellStartDate,
           CONVERT(nvarchar(30), SellStartDate) AS ConvertedDate,
           CONVERT(nvarchar(30), SellStartDate, 126) AS ISO8601FormatDate
    FROM SalesLT.Product;
    ```
* **`TRY_CAST` (Safe Conversion):** If you try to `CAST` a column that contains mixed data (e.g., converting sizes like '58' and 'S' to integers), it will crash. `TRY_CAST` converts what it can and safely returns `NULL` for the incompatible values.
    ```sql
    SELECT Name, TRY_CAST(Size AS Integer) AS NumericSize
    FROM SalesLT.Product;
    ```

---

## 3. Handling NULL Values
`NULL` means "unknown" or "missing." It is not zero, and it is not a blank space.

* **`ISNULL`:** Replaces `NULL` with a specified fallback value.
    ```sql
    -- Replaces NULL sizes with 0
    SELECT Name, ISNULL(TRY_CAST(Size AS Integer),0) AS NumericSize
    FROM SalesLT.Product;
    ```
* **`NULLIF`:** The opposite of `ISNULL`. It forces a specific value to become `NULL`.
    ```sql
    -- If the Color is 'Multi', it returns NULL instead
    SELECT Name, NULLIF(Color, 'Multi') AS SingleColor
    FROM SalesLT.Product;
    ```
* **`COALESCE`:** Evaluates a list of columns/expressions and returns the first one that is *not* `NULL`.
    ```sql
    -- Returns SellEndDate; if that is NULL, it falls back to SellStartDate
    SELECT Name, COALESCE(SellEndDate, SellStartDate) AS StatusLastUpdated
    FROM SalesLT.Product;
    ```

---

## 4. Conditional Logic: CASE Expressions
Use `CASE` expressions to apply if/then/else logic directly within your `SELECT` queries.

> **Crucial Rule for NULLs:** You cannot use `=` to check for `NULL` because comparing an unknown to an unknown results in an unknown. You must use `IS NULL` or `IS NOT NULL`.

* **Searched CASE:** Evaluates one or more specific boolean expressions. Great for checking for `NULL`.
    ```sql
    SELECT Name,
        CASE
            WHEN SellEndDate IS NULL THEN 'Currently for sale'
            ELSE 'No longer available'
        END AS SalesStatus
    FROM SalesLT.Product;
    ```
* **Simple CASE:** Compares a single column against multiple possible values.
    ```sql
    SELECT Name,
        CASE Size
            WHEN 'S' THEN 'Small'
            WHEN 'M' THEN 'Medium'
            WHEN 'L' THEN 'Large'
            WHEN 'XL' THEN 'Extra-Large'
            ELSE ISNULL(Size, 'n/a')
        END AS ProductSize
    FROM SalesLT.Product;
    ```

---

## 5. Challenge Solutions Reference
Here are the solutions to the lab challenges for easy reference and practice.

### Challenge 1: Retrieve Customer Data
* **Retrieve all customer details:**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT * FROM SalesLT.Customer;
    ```
    
</details>

* **Retrieve specific customer name data:**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT Title, FirstName, MiddleName, LastName, Suffix FROM SalesLT.Customer;
    ```

</details>

* **Retrieve names and phone numbers (Handling NULL Titles):**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT Salesperson, ISNULL(Title,'') + ' ' + LastName AS CustomerName, Phone
    FROM SalesLT.Customer;
    ```
</details>

### Challenge 2: Retrieve Customer Order Data
* **List of customer companies (ID: Name):**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT CAST(CustomerID AS varchar) + ': ' + CompanyName AS CustomerCompany
    FROM SalesLT.Customer;
    ```

</details>

* **List of sales order revisions and formatted dates:**

<details><summary>Click to view solution</summary>
    ```sql
    SELECT PurchaseOrderNumber + ' (' + STR(RevisionNumber, 1) + ')' AS OrderRevision,
           CONVERT(nvarchar(30), OrderDate, 102) AS OrderDate
    FROM SalesLT.SalesOrderHeader;
    ```
</details>

### Challenge 3: Retrieve Customer Contact Details
* **Customer names with middle names handled safely:**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT FirstName + ' ' + ISNULL(MiddleName + ' ', '') + LastName AS CustomerName
    FROM SalesLT.Customer;
    ```

</details>

* **Primary contact details (Email preferred, fallback to Phone):**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT CustomerID, COALESCE(EmailAddress, Phone) AS PrimaryContact
    FROM SalesLT.Customer;
    ```

</details>

* **Shipping status using a Searched CASE expression:**

<details><summary>Click to view solution</summary>

    ```sql
    SELECT SalesOrderID, OrderDate,
        CASE
            WHEN ShipDate IS NULL THEN 'Awaiting Shipment'
            ELSE 'Shipped'
        END AS ShippingStatus
    FROM SalesLT.SalesOrderHeader;
    ```
</details>

## Module assessment

**1. You need to retrieve the 'Name' and 'ProductNumber' columns from the 'Product' table where the 'ListPrice' is greater than 1000. Which SQL query should you use?**

<details><summary>Click to view solution</summary>

**Answer:** `SELECT Name, ProductNumber FROM Product WHERE ListPrice > 1000;`

</details>

**2. What is the primary reason for explicitly converting data types in SQL queries?**

<details><summary>Click to view solution</summary>

**Answer:** To ensure compatibility and avoid errors. *(e.g., preventing errors when trying to concatenate a numeric ID with a string name).*

</details>

**3. How can you retrieve the distinct 'Category' values from the 'Products' table?**

<details><summary>Click to view solution</summary>

**Answer:** `SELECT DISTINCT Category FROM Products;`

</details>

**4. Which SQL statement type should be used to remove a table from a database?**

<details><summary>Click to view solution</summary>

**Answer:** `DROP` *(Note: `DELETE` removes rows but keeps the table, and `TRUNCATE` removes all rows but keeps the table structure. `DROP` removes the table entirely).*

</details>

**5. What is the result of concatenating a NULL value with a string in T-SQL using the '+' operator?**

<details><summary>Click to view solution</summary>

**Answer:** The result is NULL. *(Because NULL represents an unknown value, any string concatenation with it results in NULL. You must use `ISNULL()` or `COALESCE()` to handle this).*

</details>

**6. If a SQL query is returning an error due to a missing FROM clause, which type of SQL statement is likely being used?**

<details><summary>Click to view solution</summary>

**Answer:** Data Manipulation Language (DML) *(The `SELECT` statement is used to query data and is classified under DML in this context).*

</details>

**7. Your organization needs to retrieve all columns from the 'Employee' table in the 'HR' schema of the database, but only for employees with a job title of 'Manager'. Which SQL query achieves this requirement?**

<details><summary>Click to view solution</summary>

**Answer:** `SELECT * FROM HR.Employee WHERE JobTitle = 'Manager';`

</details>

**8. A database administrator wants to create a new table in a database. Which type of SQL statement should they use?**

<details><summary>Click to view solution</summary>

**Answer:** Data Definition Language (DDL) *(DDL handles the definition and structure of database objects using commands like `CREATE`, `ALTER`, and `DROP`).*

</details>

**9. Which SQL function would you use to clean data by replacing placeholder values such as 'N/A' with NULL?**

<details><summary>Click to view solution</summary>

**Answer:** `NULLIF` *(Syntax: `NULLIF(column_name, 'N/A')` will return `NULL` if the column's value is 'N/A', otherwise it returns the actual column value).*

</details>